# Same responses different gradients

Day 22 · Chapter 13 · bounded CPU objective microscope. No training, model download or server. We hold collected responses and initial shared-decoder weights fixed, then change one declared reduction or reward-normalization rule. That isolates an objective difference from a data change. A separate bounded retry experiment accounts for rejected groups.

[Chapter](../../book/chapters/13-group-relative-policy-optimization.md) · [premeasurement specification](../../experiments/specs/2026-10-04-grpo-objective-controls.md) · [measured report](../../experiments/reports/2026-10-04-grpo-objective-controls.md)

Predict before revealing each reference. Static previews show saved reference arrays; rerun the plotting cells for your current experiment.

In [ ]:
from pathlib import Path
import sys
root=Path.cwd().resolve()
while not (root/"src"/"dongxi_llms").is_dir():
    if root.parent==root: raise RuntimeError("Open inside Dongxi_LLMs")
    root=root.parent
sys.path.insert(0,str(root/"src"))
import torch
import numpy as np
import matplotlib.pyplot as plt
from dongxi_llms.grpo_objective_controls import (
    run_controls, reduction_weights, component_advantages, objective,
    analytic_logp_gradient, clipping_fixture, SEEDS, CAP
)
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white",
                     "font.family":"DejaVu Sans","font.size":11,
                     "axes.spines.top":False,"axes.spines.right":False})
colors=["#356a9b","#b97139","#3d8268"]


## Exercise 1 Which response receives more weight

One response has one action and another has three. Both get the same positive advantage. Would averaging all valid tokens be equivalent to averaging inside each response first? What if a fixed denominator counts the declared cap rather than actual length?

Reference: the coefficients below are exact controlled numbers, not learned attention. They multiply the clipped surrogate, so they directly change its derivative.

In [ ]:
mask=torch.tensor([[True,False,False],[True,True,True]])
reductions=["response","token","fixed"]
arrays=[reduction_weights(mask,r,3).numpy() for r in reductions]
fig,axes=plt.subplots(1,3,figsize=(10.5,3.2),sharex=True,sharey=True)
for ax,array,name in zip(axes,arrays,reductions):
    im=ax.imshow(array,vmin=0,vmax=.5,cmap="Blues")
    for i in range(2):
        for j in range(3):
            ax.text(j,i,f"{array[i,j]:.3f}",ha="center",va="center",
                    color="white" if array[i,j]>.35 else "#222222")
    ax.set(xticks=range(3),xticklabels=["action 1","action 2","action 3"],
           yticks=[0,1],yticklabels=["short: 1","long: 3"],title=name+" mean")
fig.suptitle("Exact per-action coefficients; zero cells are not generated actions")
fig.tight_layout();plt.show()
print("Total response weights:",[a.sum(axis=1).tolist() for a in arrays])


The response mean gives each response total weight 1/2. The token mean gives the long response 3/4. The fixed-cap denominator is 2×3, even though only four actions exist; its total weight is 2/3. Adding padding must not change any coefficient or pretend an action was generated.

![Saved exact reduction coefficients](../figures/chapter-13/day-22-03_objective_weighting_and_filtering-01.png)

Perturbation: replace the long length with two. Keep the declared cap three and explain which denominators change.

## Exercise 2 A zero loss can still move shared weights

The policy is the same initial tiny decoder in every row. It genuinely samples EOS/0/1 under a three-action syntax grammar; no correct answer or EOS is supplied. The proxy combines answer-only correctness, complete one-numeral format and a deliberately questionable verbosity bonus. Independent quality requires the correct numeral followed by EOS.

Predict whether changing reduction can change a decoder parameter gradient even when every initial importance ratio is one. Will component-before-aggregation normalization necessarily agree with normalizing the total reward?

In [ ]:
result=run_controls()
assert result["seeds"]==list(SEEDS)
runs=result["runs"]
for run in runs:
    assert run["initial_state_sha256"]==run["final_state_sha256"]
    assert run["ratio_one_clipping_equal"]
    assert all(v["analytic_max_error"]<1e-12 for v in run["variants"])
    print(run["seed"],"first-pool tokens",run["first_attempt_pool"]["valid_tokens"],
          "actual groups",run["collection_costs"]["group_attempts"],
          "seconds",round(run["seconds"],4))
names=[v["name"] for v in runs[0]["variants"]]
fig,ax=plt.subplots(figsize=(11,4.8))
x=np.arange(len(names))
for index,(run,color) in enumerate(zip(runs,colors)):
    ax.bar(x+(index-1)*.23,[v["parameter_gradient_norm"] for v in run["variants"]],
           width=.23,color=color,label=f'seed {run["seed"]}')
labels=[n.replace("component_std","component").replace("total_std","total").replace("/","\n") for n in names]
ax.set(xticks=x,xticklabels=labels,
       ylabel="decoder parameter-gradient L2 norm",
       title="Same first-attempt responses and unchanged weights; different objectives")
ax.legend();fig.tight_layout();plt.show()


Reference: every selected-log-probability derivative is checked against an independent piecewise formula. All model weights remain unchanged: these are actual gradients, not a learning-curve claim. Centered response means have nearly zero numerical loss at initial ratio one, while their decoder gradients are nonzero. Global-token and fixed-cap gradients are collinear on the same pool but scaled differently; response means can change direction too.

![Saved actual three-seed gradient norms](../figures/chapter-13/day-22-03_objective_weighting_and_filtering-02.png)

Perturbation: use component weights ten times larger. Component-normalized advantages become ten times larger; normalizing components does not make the subsequent weights meaningless.

In [ ]:
run=runs[0]
cos=np.array(run["parameter_gradient_cosines"])
fig,ax=plt.subplots(figsize=(8.6,7.3))
im=ax.imshow(cos,vmin=-1,vmax=1,cmap="RdBu_r")
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j,i,f"{cos[i,j]:.2f}",ha="center",va="center",fontsize=8,
                color="white" if abs(cos[i,j])>.65 else "#222222")
labels=[n.replace("component_std","component").replace("total_std","total") for n in names]
ax.set(xticks=range(len(names)),xticklabels=labels,yticks=range(len(names)),
       yticklabels=labels,title=f'Actual parameter-gradient directions, seed {run["seed"]}')
plt.setp(ax.get_xticklabels(),rotation=60,ha="right")
fig.colorbar(im,ax=ax,shrink=.72,label="cosine, not performance")
fig.tight_layout();plt.show()
print("Weight-rescaling numerical error:",run["component_weight_rescaling_max_error"])


Read the cosine as a relation between update directions at this one frozen policy and sample pool. A value near one says aligned directions, not equal magnitudes or equal final behavior. These measurements cannot rank training methods.

![Saved gradient-direction comparison](../figures/chapter-13/day-22-03_objective_weighting_and_filtering-03.png)

## Exercise 3 Clipping is directional

A positive advantage should stop receiving additional encouragement above the upper boundary. A negative advantage should stop receiving additional encouragement below the lower boundary. Does enlarging only the upper clip affect both cases equally?

The next arrays are constructed log probabilities with ratios 0.6, 0.9, 1.1, 1.3 and 1.6. They are not model outputs. The actual initial model comparison has ratio one and inactive clipping.

In [ ]:
probe=clipping_fixture()
ratios=np.array(probe["ratios"])
fig,axes=plt.subplots(1,2,figsize=(10,3.6),sharey=True)
for row,ax in enumerate(axes):
    for label,color in zip(["symmetric","asymmetric"],colors):
        ax.plot(ratios,probe[label]["gradient"][row],"o-",color=color,label=label)
    ax.axhline(0,color="#888888",linewidth=.8)
    ax.axvline(.8,color="#bbbbbb",linestyle=":")
    ax.axvline(1.2,color="#bbbbbb",linestyle=":")
    ax.axvline(1.4,color="#aaaaaa",linestyle="--")
    ax.set(xlabel="chosen-action importance ratio",
           title="positive advantage" if row==0 else "negative advantage")
axes[0].set_ylabel("derivative of minimization loss wrt log p")
axes[1].legend();fig.tight_layout();plt.show()
print("Positive derivative at ratio1.3:",probe["symmetric"]["gradient"][0][3],
      probe["asymmetric"]["gradient"][0][3])


Reference: increasing the upper boundary from 1.2 to 1.4 restores positive-advantage pressure at ratio 1.3. It does not move the negative-advantage lower boundary. A negative derivative means gradient descent increases that selected log probability.

![Saved explicitly constructed clipping derivatives](../figures/chapter-13/day-22-03_objective_weighting_and_filtering-04.png)

Perturbation: change the lower clip instead. Keep the sign interpretation and distinguish clipped surrogate behavior from a guarantee about the full vocabulary distribution.

## Exercise 4 Rejecting groups does not make decoding free

We retry only prompts with no mixed independent-quality group, at most three groups per prompt. Every all-wrong or all-correct rejected group remains in the raw ledger. This experiment is separate from the objective comparison, which always uses the original first-attempt pool.

Predict whether equal selected response counts imply equal collection cost.

In [ ]:
fig,(token_ax,group_ax)=plt.subplots(1,2,figsize=(10.5,3.8))
x=np.arange(len(runs))
attempted=[r["collection_costs"]["valid_tokens_attempted"] for r in runs]
selected=[r["collection_costs"]["valid_tokens_selected"] for r in runs]
token_ax.bar(x-.18,attempted,.36,color=colors[0],label="all attempted")
token_ax.bar(x+.18,selected,.36,color=colors[2],label="selected only")
token_ax.set(xticks=x,xticklabels=[str(r["seed"]) for r in runs],
             xlabel="fixed seed",ylabel="valid generated actions, EOS included",
             title="Rejected groups still consume generation")
token_ax.legend()
group_ax.bar(x,[r["collection_costs"]["group_attempts"] for r in runs],
             color=colors[1],label="attempted groups")
group_ax.axhline(5,color=colors[2],linestyle="--",label="five selected groups")
group_ax.set(xticks=x,xticklabels=[str(r["seed"]) for r in runs],xlabel="fixed seed",
             ylabel="groups of eight responses",title="Bounded retries are not equal cost")
group_ax.legend();fig.tight_layout();plt.show()
for r in runs:
    print(r["seed"],r["collection_costs"])
assert sum(attempted)>sum(selected)


Reference: all three seeds eventually supply five mixed groups, but consume different numbers of attempted groups and actions. The all-constant test deliberately exhausts the retry cap and selects nothing; that diagnostic prevents the successful current seeds from becoming an assumed guarantee.

Generation full-prefix position counts include finished rows still processed by this simple uncached batch. They exclude additional collection/analysis rescoring. They are not FLOPs or a serving-throughput benchmark.

![Saved actual attempted and selected costs](../figures/chapter-13/day-22-03_objective_weighting_and_filtering-05.png)

Deep checkpoint: would a smaller filtered training loss establish better reasoning? No. Filtering changes the sampled training population and spends extra collection work. An independent held-out evaluation and matched budget would be needed for a capability claim. No optimizer update occurs in this microscope.